## Structured Output

### What it is
Instead of returning free text, the model returns data in a **fixed shape**
(schema) you define — like a form it must fill in.

### Why use it
- Output is **predictable** and easy to parse.
- No messy text cleanup — you get clean fields.
- Great for feeding results into the **next step** (code, database, API).

### Pydantic
You define the shape using a **Pydantic model** (a Python class that describes
the fields and their types).

```python
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
    city: str
```

### Use it with a model
`with_structured_output()` forces the model to return that shape.

```python
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("John is 30 and lives in Pune.")
print(result)
# Person(name='John', age=30, city='Pune')
```

### Access fields directly
```python
print(result.name)   # John
print(result.age)    # 30
```

### Key notes
- The **Pydantic class** defines the fields and types.
- The model must return data matching that schema.
- You get a **Python object**, not a string — access with `result.field`.
- Types (`str`, `int`, etc.) are validated automatically.

In [11]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llama_model = ChatOpenAI(
    model="llama3.2",
    base_url=os.getenv("GE_BASE_URL"),
    api_key=os.getenv("GE_API_KEY"),
    temperature=0,
)
llama_model


ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000028344CE1D90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000028344CE12E0>, root_client=<openai.OpenAI object at 0x0000028344CE1CD0>, root_async_client=<openai.AsyncOpenAI object at 0x0000028344CE04A0>, model_name='llama3.2', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://genai-imaging-lab.apps.ge-healthcare.net:4000/v1', openai_proxy=None, stream_chunk_timeout=120.0)

#### Import Pydantic

In [12]:
from pydantic import BaseModel, Field

In [13]:
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie (0-10)")

In [16]:
# Without structured output
response = llama_model.invoke("Provide details of the movie Inception")
response

AIMessage(content='Inception is a 2010 science fiction action film written, directed, and produced by Christopher Nolan. The film stars Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe, Dileep Rao, Cillian Murphy, and Marion Cotillard.\n\n**Plot:**\n\nThe story follows Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" - planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name and allow him to return to the United States to see his children.\n\nTo accomplish this task, Cobb assembles a team of experts: Arthur (Joseph Gordon-Levitt), a point man; Ariadne (Ellen Page), an architect who designs the dreamscapes

In [17]:
response.usage_metadata

{'input_tokens': 32,
 'output_tokens': 696,
 'total_tokens': 728,
 'input_token_details': {},
 'output_token_details': {}}

In [22]:
model_with_structure = llama_model.with_structured_output(Movie, include_raw=True)
model_with_structure

{
  raw: _ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000028344CE1D90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000028344CE12E0>, root_client=<openai.OpenAI object at 0x0000028344CE1CD0>, root_async_client=<openai.AsyncOpenAI object at 0x0000028344CE04A0>, model_name='llama3.2', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://genai-imaging-lab.apps.ge-healthcare.net:4000/v1', openai_proxy=None, stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Th

In [26]:
structured_response = model_with_structure.invoke("Provide details of the movie Inception")
structured_response

{'raw': AIMessage(content='{"title": "Inception", "year": 2010, "genre": "Science Fiction, Action, Thriller", "director": "Christopher Nolan", "rating": 8.5}\n\n   ', additional_kwargs={'parsed': Movie(title='Inception', year=2010, genre='Science Fiction, Action, Thriller', director='Christopher Nolan', rating=8.5), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 32, 'total_tokens': 76, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama_chat/llama3.2', 'system_fingerprint': None, 'id': 'chatcmpl-8cc2a18c-77da-431b-9a56-7e5a6a7ab1e7', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fbc72-2b47-70f0-a8e1-481a1522acec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 44, 'total_tokens': 76, 'input_token_details': {}, 'output_token_details': {}}),
 'parsed': Movie(title='Inception', year=2010, genre='Science Fiction, Action,

In [24]:
structured_response["raw"].usage_metadata

{'input_tokens': 32,
 'output_tokens': 44,
 'total_tokens': 76,
 'input_token_details': {},
 'output_token_details': {}}

### NOTE : structured call generated far fewer tokens 

## `include_raw` in Structured Output

### What it is
An option in `with_structured_output()` that decides **what you get back** —
just the parsed object, or the parsed object **plus** the raw model message.

### Default (`include_raw=False`)
Returns ONLY your parsed Pydantic object.

```python
model_with_structure = model.with_structured_output(Movie)
result = model_with_structure.invoke("Details of Inception")

print(result)         # Movie(title='Inception', year=2010, ...)
print(result.title)   # Inception
```

❌ No access to token usage or metadata.

### With `include_raw=True`
Returns a **dictionary** with three keys: the raw message, the parsed object,
and any parsing error.

```python
model_with_structure = model.with_structured_output(Movie, include_raw=True)
result = model_with_structure.invoke("Details of Inception")
```

Result shape:
```python
{
    "raw": AIMessage(...),     # full message → has usage_metadata
    "parsed": Movie(...),      # your structured object
    "parsing_error": None,     # error info if parsing failed
}
```

### Access the parts
```python
print(result["parsed"])                 # Movie(title='Inception', ...)
print(result["parsed"].title)           # Inception
print(result["raw"].usage_metadata)     # {'input_tokens': ..., 'total_tokens': ...}
print(result["parsing_error"])          # None (or the error)
```

### When to use
| Use `include_raw=True` when you need… |
|---------------------------------------|
| Token usage / cost tracking (`usage_metadata`) |
| Response metadata (model, headers) |
| To safely handle parsing failures (`parsing_error`) |

### When NOT to use
- If you only need the clean data → keep default (`include_raw=False`).

### Key notes
- Default → returns the **object** directly.
- `include_raw=True` → returns a **dict** (`raw`, `parsed`, `parsing_error`).
- Token usage lives on `result["raw"].usage_metadata`.
- Safer for production — `parsing_error` lets you catch bad outputs.

## Nested Strucrture

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")

#Actor class is nested in Movie class to represent the actors in the movie
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie (0-10)")
    actors: list[Actor] = Field(..., description="List of actors in the movie")
    budget: float = Field(..., description="The budget of the movie in USD")